<a href="https://colab.research.google.com/github/hincaltopcuoglu/Npath-text-mining/blob/master/npath_text_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NPath Text Mining: N-Gram Analysis for Classification

This notebook performs n-gram analysis and discriminative feature extraction for text classification using the opinions dataset.

**Target**: `type` column (Claim, Evidence, Counterclaim, etc.)  
**Features**: `text` column (student opinions/arguments)

## Analysis Pipeline:
1. Load and clean opinions dataset
2. Generate n-grams (1-grams through 5-grams)
3. Calculate discriminative scores for classification
4. Pattern ranking and analysis
5. Sequential pattern visualization (Sankey diagrams)
6. Export results for model training

---
**Dataset**: ~34K student opinion texts  
**GitHub**: https://github.com/hincaltopcuoglu/Npath-text-mining

In [10]:
# @title 🔄 FORCE SYNC: Get Latest Code
# Download latest code from GitHub

import os
from pathlib import Path
import urllib.request
import urllib.error

print("🔄 Force Sync: Getting latest code from GitHub...")

# Try direct file downloads first (more reliable)
print("📥 Trying direct file downloads...")

# Files to download
files_to_download = [
    ("colab_ngram_analysis.py", "https://raw.githubusercontent.com/hincaltopcuoglu/Npath-text-mining/master/colab_ngram_analysis.py"),
    ("pattern_ranking.py", "https://raw.githubusercontent.com/hincaltopcuoglu/Npath-text-mining/master/pattern_ranking.py"),
    ("sankey_visualizer.py", "https://raw.githubusercontent.com/hincaltopcuoglu/Npath-text-mining/master/sankey_visualizer.py"),
    ("COLAB_SETUP.md", "https://raw.githubusercontent.com/hincaltopcuoglu/Npath-text-mining/master/COLAB_SETUP.md"),
]

direct_download_success = 0
direct_download_failed = 0

for filename, url in files_to_download:
    try:
        # Always download to get latest version (force update)
        urllib.request.urlretrieve(url, filename)
        print(f"  ✅ Downloaded/Updated {filename}")
        direct_download_success += 1
    except Exception as e:
        print(f"  ❌ Failed {filename}: {str(e)[:50]}...")
        direct_download_failed += 1

# Verify files
required_files = ["colab_ngram_analysis.py", "pattern_ranking.py", "sankey_visualizer.py"]
print("\n🔍 File verification:")
missing_files = []
for file_path in required_files:
    if Path(file_path).exists():
        print(f"  ✅ {file_path}")
    else:
        print(f"  ❌ Missing: {file_path}")
        missing_files.append(file_path)

if missing_files:
    print("\n⚠️  Some files missing. Use the MANUAL DOWNLOAD cell below.")
else:
    print("\n🎯 All required files present - ready to run analysis!")

print(f"\n📊 Summary: {direct_download_success} files downloaded, {direct_download_failed} failed")

🔄 Force Sync: Getting latest code from GitHub...
📥 Trying direct file downloads...
  ✅ Downloaded/Updated colab_ngram_analysis.py
  ✅ Downloaded/Updated pattern_ranking.py
  ✅ Downloaded/Updated sankey_visualizer.py
  ✅ Downloaded/Updated COLAB_SETUP.md

🔍 File verification:
  ✅ colab_ngram_analysis.py
  ✅ pattern_ranking.py
  ✅ sankey_visualizer.py

🎯 All required files present - ready to run analysis!

📊 Summary: 4 files downloaded, 0 failed


In [11]:
# @title 🔄 Quick Sync: Check for Updates
# Check if notebook or code files have been updated on GitHub
import os
from pathlib import Path
import urllib.request
import json

# GitHub repository details
GITHUB_USERNAME = "hincaltopcuoglu"  # @param {type:"string"}
REPO_NAME = "Npath-text-mining"     # @param {type:"string"}
BRANCH = "master"                   # @param {type:"string"}

print("🔍 Checking for updates...")

# Check if notebook file was updated
notebook_url = f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/{REPO_NAME}/{BRANCH}/npath_text_analysis.ipynb"
try:
    response = urllib.request.urlopen(notebook_url)
    remote_notebook = json.loads(response.read())
    remote_cell_count = len(remote_notebook['cells'])

    print(f"📊 Remote notebook has {remote_cell_count} cells")

    # Check key files
    key_files = [
        "sankey_visualizer.py",
        "colab_ngram_analysis.py",
        "pattern_ranking.py"
    ]

    print("\n📁 Checking key files:")
    for filename in key_files:
        file_url = f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/{REPO_NAME}/{BRANCH}/{filename}"
        try:
            response = urllib.request.urlopen(file_url)
            remote_size = len(response.read())
            local_size = Path(filename).stat().st_size if Path(filename).exists() else 0

            if local_size == 0:
                print(f"  ⚠️  {filename}: NOT FOUND locally - Run FORCE SYNC!")
            elif abs(remote_size - local_size) > 100:  # Allow small differences
                print(f"  🔄 {filename}: Size differs - Run FORCE SYNC to update")
            else:
                print(f"  ✅ {filename}: Up to date")
        except:
            print(f"  ❓ {filename}: Could not check")

    print("\n💡 IMPORTANT:")
    print("   • If NEW CELLS were added → REFRESH the page (F5) to see them")
    print("   • If only CODE changed → Run FORCE SYNC cell (no refresh needed)")
    print("   • Always run FORCE SYNC after refreshing to get latest files")

except Exception as e:
    print(f"⚠️  Could not check for updates: {e}")
    print("💡 Run FORCE SYNC cell to get latest code")

🔍 Checking for updates...
📊 Remote notebook has 16 cells

📁 Checking key files:
  ✅ sankey_visualizer.py: Up to date
  ✅ colab_ngram_analysis.py: Up to date
  ✅ pattern_ranking.py: Up to date

💡 IMPORTANT:
   • If NEW CELLS were added → REFRESH the page (F5) to see them
   • If only CODE changed → Run FORCE SYNC cell (no refresh needed)
   • Always run FORCE SYNC after refreshing to get latest files


In [12]:
# @title 📦 Install Dependencies
# Install required packages for Colab
print("📦 Installing dependencies...")

# Core ML/data science packages
!pip install -q pandas numpy scikit-learn matplotlib seaborn plotly kaleido

# NLP packages
!pip install -q nltk tqdm

# Download NLTK data
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("✅ Dependencies installed!")

# Test imports
try:
    import pandas as pd
    import numpy as np
    from sklearn.feature_extraction.text import CountVectorizer
    from nltk.util import ngrams
    import plotly.graph_objects as go
    print("✅ All imports successful!")
except ImportError as e:
    print(f"❌ Import error: {e}")

📦 Installing dependencies...
✅ Dependencies installed!
✅ All imports successful!


In [13]:
# @title 💾 Setup Persistence: Save to Google Drive
# Mount Google Drive for persistent storage across sessions

from google.colab import drive
import os
from pathlib import Path

print("💾 Setting up Google Drive persistence...")

# Mount Google Drive
drive.mount('/content/drive', force_remount=False)

# Create persistent directories
persistent_dir = Path('/content/drive/MyDrive/NPath_Analysis')
results_backup_dir = persistent_dir / 'results_backup'

persistent_dir.mkdir(exist_ok=True)
results_backup_dir.mkdir(exist_ok=True)

print(f"✅ Google Drive mounted successfully!")
print(f"📁 Persistent storage: {persistent_dir}")
print(f"💾 Results backup: {results_backup_dir}")
print(f"\n🎯 Your analysis results will be saved to Google Drive!")
print(f"💡 You can resume work from any Colab session.")

💾 Setting up Google Drive persistence...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully!
📁 Persistent storage: /content/drive/MyDrive/NPath_Analysis
💾 Results backup: /content/drive/MyDrive/NPath_Analysis/results_backup

🎯 Your analysis results will be saved to Google Drive!
💡 You can resume work from any Colab session.


In [16]:
# @title 📊 Quick Data Check
# Load and examine the opinions dataset
import pandas as pd
from collections import Counter

print("🔍 Loading opinions dataset...")

# Load data
try:
    df = pd.read_csv('/content/drive/MyDrive/opinions.csv',
                     sep=',',
                     quotechar='"',
                     escapechar='\\',
                     on_bad_lines='skip',
                     engine='python')

    # Clean column names
    df.columns = df.columns.str.replace(';;;;;;', '')

    # Clean data
    initial_rows = len(df)
    df = df.dropna(subset=['text', 'type'])
    df = df[df['text'].str.len() > 10]
    df = df[df['type'].str.len() > 0]

    print(f"✅ Loaded {len(df)} rows from {initial_rows} total")
    print(f"📊 Columns: {list(df.columns)}")
    print(f"🎯 Target classes: {df['type'].nunique()}")

    # Show class distribution
    print("\n📈 Class Distribution (Top 10):")
    class_counts = df['type'].value_counts()
    for cls, count in class_counts.head(10).items():
        print(f"  {cls[:40]:<40}: {count}")

    # Show sample texts
    print("\n📝 Sample Texts:")
    for i, row in df.head(3).iterrows():
        print(f"  {row['type'][:15]:<15}: {row['text'][:80]}...")

except Exception as e:
    print(f"❌ Error loading data: {e}")
    print("💡 Make sure /content/drive/MyDrive/opinions.csv exists. Run FORCE SYNC if needed.")

🔍 Loading opinions dataset...
❌ Error loading data: [Errno 2] No such file or directory: 'content/drive/MyDrive/opinions.csv'
💡 Make sure /content/drive/MyDrive/opinions.csv exists. Run FORCE SYNC if needed.


In [6]:
# @title 🚀 Run N-Gram Analysis
# Execute the complete n-gram analysis pipeline

# Import the analyzer
from colab_ngram_analysis import ColabNgramAnalyzer

# Configuration parameters
N_VALUES = [1, 2, 3, 4, 5]  # @param {type:"raw"} # Include 1-grams and 5-grams for Sankey visualization
MIN_FREQ = 5                # @param {type:"integer"} # Minimum frequency for n-grams
MIN_SUPPORT = 10            # @param {type:"integer"} # Minimum support for discriminative analysis
TOP_K = 500                 # @param {type:"integer"} # Top k discriminative n-grams per class
BATCH_SIZE = 1000           # @param {type:"integer"} # Processing batch size

print("🚀 Starting N-Gram Analysis Pipeline")
print("=" * 50)
print(f"N-grams: {N_VALUES}")
print(f"Min frequency: {MIN_FREQ}")
print(f"Min support: {MIN_SUPPORT}")
print(f"Top k per class: {TOP_K}")
print(f"Batch size: {BATCH_SIZE}")
print("=" * 50)

# Initialize analyzer
analyzer = ColabNgramAnalyzer(
    data_path='data/raw/opinions.csv',
    text_col='text',
    target_col='type'
)

# Run complete analysis
analyzer.run_complete_analysis(
    n_values=N_VALUES,
    min_freq=MIN_FREQ,
    min_support=MIN_SUPPORT,
    top_k=TOP_K,
    batch_size=BATCH_SIZE
)

print("\n✅ Analysis Complete!")

✅ NLTK data downloaded successfully
🚀 Starting N-Gram Analysis Pipeline
N-grams: [1, 2, 3, 4, 5]
Min frequency: 5
Min support: 10
Top k per class: 500
Batch size: 1000
🚀 Starting Complete N-Gram Analysis for Opinions Dataset
🔄 Loading and cleaning data...
Standard parsing failed: [Errno 2] No such file or directory: 'data/raw/opinions.csv'


Exception: Both parsing methods failed: [Errno 2] No such file or directory: 'data/raw/opinions.csv', [Errno 2] No such file or directory: 'data/raw/opinions.csv'

In [7]:
# @title 📁 Check Results
# Examine the generated results and files
import os
from pathlib import Path
import pandas as pd

results_dir = Path('colab_results')
if results_dir.exists():
    print(f"📂 Results directory: {results_dir.absolute()}")

    # List all files
    files = list(results_dir.glob('*'))
    print(f"📄 Generated files: {len(files)}")

    for file_path in sorted(files):
        size_mb = file_path.stat().st_size / (1024 * 1024)
        print(f"  📄 {file_path.name:<30}: {size_mb:.2f} MB")

    print("\n🔍 Sample of discriminative 2-grams:")

    # Load and show sample discriminative results
    disc_file = results_dir / '2gram_discriminative.csv'
    if disc_file.exists():
        disc_df = pd.read_csv(disc_file)
        print(f"\nTotal discriminative 2-grams: {len(disc_df)}")

        # Show top discriminative for each class
        for class_name in disc_df['class'].unique()[:5]:  # Show first 5 classes
            class_data = disc_df[disc_df['class'] == class_name]
            top_ngrams = class_data.nlargest(3, 'discriminative_score')
            print(f"\n🎯 Top 2-grams for '{class_name}':")
            for _, row in top_ngrams.iterrows():
                print(f"  {row['discriminative_score']:.1f}x: '{row['ngram'][:50]}...'")

else:
    print("❌ Results directory not found. Run the analysis first.")

# Check for visualization files
viz_files = list(Path('.').glob('colab_results/*.png'))
if viz_files:
    print(f"\n🖼️  Visualization files: {len(viz_files)}")
    for viz_file in viz_files:
        print(f"  🖼️  {viz_file.name}")

❌ Results directory not found. Run the analysis first.


In [ ]:
# @title 🎯 Advanced Pattern Ranking & Analysis
# Rank patterns by confidence, lift, and rarity
# Uses multiple scoring metrics to identify the most discriminative patterns

from pattern_ranking import PatternRanker

print("🎯 Starting Advanced Pattern Ranking Analysis")
print("=" * 60)

# Initialize ranker
ranker = PatternRanker(results_dir='colab_results')

# Run complete ranking analysis (this handles everything: loading, ranking, exporting, visualizing)
ranker.run_complete_ranking_analysis(
    n_values=[2, 3, 4],  # Analyze bigrams, trigrams, 4-grams
    top_k=100            # Top 100 patterns per class per n-gram type
)

print("\n✅ Pattern Ranking Complete!")
print("\n📊 Output Files:")
print("   • Xgram_rankings.csv - Ranked patterns with confidence, lift, rarity scores")
print("   • top_Xgram_patterns_comparison.png - Visualizations of top patterns")

🎯 Starting Advanced Pattern Ranking Analysis
🚀 Starting Complete Pattern Ranking Analysis
📥 Loading pattern data from results...
  ✅ Loaded 2-gram counts: 2090 entries
  ✅ Loaded 2-gram discriminative: 229 entries
  ✅ Loaded 3-gram counts: 414 entries
  ✅ Loaded 3-gram discriminative: 9 entries
  ✅ Loaded 4-gram counts: 158 entries
📊 Loaded pattern data for 2 n-gram types
📊 Calculating class statistics...
🎯 Found 3 classes
📈 Class document counts:
.1f
.1f
.1f
.1f

🔍 Analyzing 2-gram patterns...

🎯 PATTERN RANKING REPORT - 2-GRAMS
📊 Classes analyzed: 1
🎯 Top 100 patterns per class
📈 Scoring metrics: Confidence, Lift, Rarity, Discriminative, Combined

🏆 Class: Rebuttal
--------------------------------------------------
2d

  ✅ Visualization saved: top_2gram_patterns_comparison.png

🔍 Analyzing 3-gram patterns...

🎯 PATTERN RANKING REPORT - 3-GRAMS
❌ No valid 3-gram patterns found
❌ No patterns found for 3-grams
❌ No valid 3-gram patterns found

🔍 Analyzing 4-gram patterns...

🎯 PATTERN R

In [ ]:
# @title 🌊 Sankey Diagram: Complete N-gram Flow → Target Classes
# Create comprehensive Sankey diagram showing ALL n-grams and ALL classes in one view
# Shows: 1-gram → 2-gram → 3-gram → 4-gram → 5-gram → Target Classes

from sankey_visualizer import SankeyVisualizer

print("🌊 Creating Comprehensive Sankey Diagram - All Patterns & Classes")
print("=" * 80)
print("📊 This diagram shows:")
print("   • All n-gram levels (1-gram through 5-gram)")
print("   • All target classes at the rightmost position")
print("   • Sequential flows between n-grams")
print("   • Discriminative flows from n-grams to their target classes")
print("=" * 80)

# Initialize visualizer
sankey = SankeyVisualizer(results_dir='colab_results')

# Load n-gram data (including 1-grams and 5-grams)
sankey.load_ngram_data(n_values=[1, 2, 3, 4, 5])

# Create comprehensive Sankey diagram showing ALL classes and MORE patterns
print("\n📊 Generating comprehensive Sankey diagram...")
print("   Showing up to 30 patterns per class per n-gram level for complete coverage")

fig = sankey.create_sankey_diagram(
    top_k_per_class=30,  # Increased to show more patterns - ALL visible in one diagram
    output_file='sankey_npath_complete_view.html'
)

if fig:
    print("\n✅ Comprehensive Sankey diagram created successfully!")
    print("📂 File: colab_results/sankey_npath_complete_view.html")
    print("\n💡 Diagram Features:")
    print("   • Left columns: N-grams (1-gram → 2-gram → 3-gram → 4-gram → 5-gram)")
    print("   • Rightmost column: ALL target classes")
    print("   • Horizontal flows: Sequential n-gram evolution")
    print("   • Final flows: N-grams → Target Classes (discriminative patterns)")
    print("   • Flow width: Pattern importance/discriminative score")
    print("   • Colors: Different color per class")
    print("\n🌐 Open the HTML file in your browser to view the interactive diagram")
    print("   You can zoom, pan, and hover over nodes/flows for details")
    print("   All patterns and classes are visible in ONE comprehensive view!")
else:
    print("⚠️  Could not create Sankey diagram. Make sure n-gram analysis completed successfully.")
    print("💡 Run the 'Run N-Gram Analysis' cell first with N_VALUES = [1, 2, 3, 4, 5]")

🌊 Creating Comprehensive Sankey Diagram - All Patterns & Classes
📊 This diagram shows:
   • All n-gram levels (1-gram through 5-gram)
   • All target classes at the rightmost position
   • Sequential flows between n-grams
   • Discriminative flows from n-grams to their target classes
📥 Loading n-gram data for Sankey visualization...
  ✅ Loaded 1-gram counts: 3145 entries
  ✅ Loaded 1-gram discriminative: 740 entries
  ✅ Loaded 2-gram counts: 2090 entries
  ✅ Loaded 2-gram discriminative: 229 entries
  ✅ Loaded 3-gram counts: 414 entries
  ✅ Loaded 3-gram discriminative: 9 entries
  ✅ Loaded 4-gram counts: 158 entries
  ✅ Loaded 5-gram counts: 83 entries
📊 Loaded data for 5 n-gram types

📊 Generating comprehensive Sankey diagram...
   Showing up to 30 patterns per class per n-gram level for complete coverage

CREATING SANKEY DIAGRAM - nPath Sequential Pattern Visualization

🔄 Extracting sequential n-gram patterns...
  Found 3 classes: ['Counterclaim', 'Evidence', 'Rebuttal']
  N-gram se

ValueError: 
    Invalid element(s) received for the 'color' property of sankey.link
        Invalid elements include: ['#1f77b480', '#1f77b480', '#ff7f0e80', '#ff7f0e80', '#ff7f0e80', '#ff7f0e80', '#1f77b480']

    The 'color' property is a color and may be specified as:
      - A hex string (e.g. '#ff0000')
      - An rgb/rgba string (e.g. 'rgb(255,0,0)')
      - An hsl/hsla string (e.g. 'hsl(0,100%,50%)')
      - An hsv/hsva string (e.g. 'hsv(0,100%,100%)')
      - A named CSS color:
            aliceblue, antiquewhite, aqua, aquamarine, azure,
            beige, bisque, black, blanchedalmond, blue,
            blueviolet, brown, burlywood, cadetblue,
            chartreuse, chocolate, coral, cornflowerblue,
            cornsilk, crimson, cyan, darkblue, darkcyan,
            darkgoldenrod, darkgray, darkgrey, darkgreen,
            darkkhaki, darkmagenta, darkolivegreen, darkorange,
            darkorchid, darkred, darksalmon, darkseagreen,
            darkslateblue, darkslategray,

In [ ]:
# @title 💾 Download Results
# Create zip file and prepare for download
import shutil
from google.colab import files
from pathlib import Path

results_dir = 'colab_results'
zip_filename = 'npath_ngram_results.zip'

if Path(results_dir).exists():
    print(f"📦 Creating zip archive: {zip_filename}")
    shutil.make_archive('npath_ngram_results', 'zip', results_dir)

    # Get file size
    zip_size = Path(zip_filename).stat().st_size / (1024 * 1024)
    print(f"✅ Archive created: {zip_size:.2f} MB")

    # Download
    print("\n⬇️  Starting download...")
    files.download(zip_filename)

else:
    print("❌ No results to download. Run the analysis first.")

# Also offer individual file downloads
print("\n📄 Individual file downloads:")
if Path(results_dir).exists():
    csv_files = list(Path(results_dir).glob('*.csv'))
    html_files = list(Path(results_dir).glob('*.html'))
    for csv_file in csv_files:
        print(f"  - {csv_file.name}")
    for html_file in html_files:
        print(f"  - {html_file.name}")
    # Uncomment to download individual files:
    # files.download(str(csv_file))

In [ ]:
# @title 🔍 Verify Sankey Visualizer Module
# Check if sankey_visualizer.py is correctly imported and has the right colors

import importlib
import sys

# Remove old module if cached
if 'sankey_visualizer' in sys.modules:
    del sys.modules['sankey_visualizer']
    print("🔄 Cleared cached sankey_visualizer module")

# Import fresh
from sankey_visualizer import SankeyVisualizer

# Check the safe_colors palette in the module
print("📋 Checking sankey_visualizer.py source...")

import inspect
source = inspect.getsource(SankeyVisualizer.extract_sequential_patterns)

# Look for safe_colors in the source
if "safe_colors = [" in source:
    print("✅ Found safe_colors palette in source code")

    # Extract and display the colors
    lines = source.split('\n')
    for i, line in enumerate(lines):
        if 'safe_colors = [' in line:
            print("\n🎨 Safe Colors Palette:")
            j = i
            while j < len(lines) and ']' not in lines[j]:
                if '#' in lines[j]:
                    print(f"   {lines[j].strip()}")
                j += 1
            if j < len(lines):
                print(f"   {lines[j].strip()}")
            break
else:
    print("❌ ERROR: safe_colors palette NOT found in source code!")
    print("   The file sankey_visualizer.py may not be the updated version")

# Verify the module path
print(f"\n📂 Module loaded from: {inspect.getfile(SankeyVisualizer)}")

# Test color initialization
print("\n🧪 Testing color initialization...")
try:
    sankey = SankeyVisualizer(results_dir='colab_results')
    print(f"✅ SankeyVisualizer instance created successfully")
    print(f"✅ Initial class_colors dict: {sankey.class_colors}")
except Exception as e:
    print(f"❌ Error creating SankeyVisualizer: {e}")

📋 Checking sankey_visualizer.py source...
✅ Found safe_colors palette in source code

🎨 Safe Colors Palette:
   '#1f77b4',  # Blue
   '#ff7f0e',  # Orange
   '#2ca02c',  # Green
   '#d62728',  # Red
   '#9467bd',  # Purple
   '#8c564b',  # Brown
   '#e377c2',  # Pink
   '#7f7f7f',  # Gray
   '#bcbd22',  # Olive
   '#17becf',  # Cyan
   '#aec7e8',  # Light Blue
   '#ffbb78',  # Light Orange
   '#98df8a',  # Light Green
   '#ff9896',  # Light Red
   '#c5b0d5',  # Light Purple
   '#c49c94',  # Light Brown
   '#f7b6d2',  # Light Pink
   '#c7c7c7',  # Light Gray
   '#dbbd22',  # Darker Olive
   '#9edae5',  # Light Cyan
   ]

📂 Module loaded from: /Users/hincaltopcuoglu/Npath-text-mining/sankey_visualizer.py

🧪 Testing color initialization...
✅ SankeyVisualizer instance created successfully
✅ Initial class_colors dict: {}


In [ ]:
# @title 🌊 Sankey Diagram: Complete N-gram Flow → Target Classes (FIXED)
# Create comprehensive Sankey diagram showing ALL n-grams and ALL classes in one view
# Shows: 1-gram → 2-gram → 3-gram → 4-gram → 5-gram → Target Classes

import sys
import importlib

# Force reload of the module to ensure latest version is used
if 'sankey_visualizer' in sys.modules:
    del sys.modules['sankey_visualizer']

from sankey_visualizer import SankeyVisualizer

print("🌊 Creating Comprehensive Sankey Diagram - All Patterns & Classes")
print("=" * 80)
print("📊 This diagram shows:")
print("   • All n-gram levels (1-gram through 5-gram)")
print("   • All target classes at the rightmost position")
print("   • Sequential flows between n-grams")
print("   • Discriminative flows from n-grams to their target classes")
print("=" * 80)

# Initialize visualizer
sankey = SankeyVisualizer(results_dir='colab_results')

# Load n-gram data (including 1-grams and 5-grams)
sankey.load_ngram_data(n_values=[1, 2, 3, 4, 5])

# Create comprehensive Sankey diagram showing ALL classes and MORE patterns
print("\n📊 Generating comprehensive Sankey diagram...")
print("   Showing up to 30 patterns per class per n-gram level for complete coverage")

fig = sankey.create_sankey_diagram(
    top_k_per_class=30,  # Increased to show more patterns - ALL visible in one diagram
    output_file='sankey_npath_complete_view.html'
)

if fig:
    print("\n✅ Comprehensive Sankey diagram created successfully!")
    print("📂 File: colab_results/sankey_npath_complete_view.html")
    print("\n💡 Diagram Features:")
    print("   • Left columns: N-grams (1-gram → 2-gram → 3-gram → 4-gram → 5-gram)")
    print("   • Rightmost column: ALL target classes")
    print("   • Horizontal flows: Sequential n-gram evolution")
    print("   • Final flows: N-grams → Target Classes (discriminative patterns)")
    print("   • Flow width: Pattern importance/discriminative score")
    print("   • Colors: Different color per class")
    print("\n🌐 Open the HTML file in your browser to view the interactive diagram")
    print("   You can zoom, pan, and hover over nodes/flows for details")
    print("   All patterns and classes are visible in ONE comprehensive view!")
else:
    print("⚠️  Could not create Sankey diagram. Make sure n-gram analysis completed successfully.")
    print("💡 Run the 'Run N-Gram Analysis' cell first with N_VALUES = [1, 2, 3, 4, 5]")

In [ ]:
# @title 💾 Auto-Save Results to Drive
# Automatically save results to Google Drive for persistence

import shutil
from pathlib import Path

results_dir = Path('colab_results')
drive_backup_dir = Path('/content/drive/MyDrive/NPath_Analysis/results_backup')

if results_dir.exists():
    print("💾 Saving results to Google Drive...")

    # Ensure backup directory exists
    drive_backup_dir.mkdir(parents=True, exist_ok=True)

    # Copy all results
    for file_path in results_dir.glob('*'):
        if file_path.is_file():
            dest_path = drive_backup_dir / file_path.name
            shutil.copy2(file_path, dest_path)
            print(f"  ✅ Saved: {file_path.name}")

    print(f"\n✅ All results saved to: {drive_backup_dir}")
    print("💡 You can resume analysis from any Colab session!")
else:
    print("❌ No results to save. Run the analysis first.")

# 📋 Usage Instructions

## Quick Start:
1. **Run FORCE SYNC** to get latest code from GitHub
2. **Install Dependencies** (if first time)
3. **Setup Persistence** to mount Google Drive (optional but recommended)
4. **Run N-Gram Analysis** - this takes 10-30 minutes
5. **Check Results** to see what was generated
6. **Pattern Ranking** to rank patterns by importance
7. **Sankey Diagram** to visualize sequential patterns
8. **Download Results** or **Auto-Save to Drive**

## Output Files:
- `Xgram_counts.csv`: Raw n-gram frequencies by class
- `Xgram_discriminative.csv`: Discriminative scores for classification
- `Xgram_rankings.csv`: Pattern rankings with confidence, lift, rarity
- `top_Xgram_discriminative.png`: Visualization of top discriminative n-grams
- `top_Xgram_patterns_comparison.png`: Pattern ranking comparisons
- `sankey_npath_sequential_flow.html`: Interactive Sankey diagram

## Next Steps:
1. Use discriminative n-grams as features for classification models
2. Train ML models (SVM, Random Forest, BERT) using these features
3. Compare performance across different n-gram types
4. Analyze sequential patterns using Sankey diagrams

---
**Happy analyzing! 🚀**

# DistilBERT Semantic Pattern Analysis

Extract semantic themes using DistilBERT embeddings and visualize with Sankey.

In [ ]:
# @title 📥 Install DistilBERT
import subprocess, sys
for pkg in ['torch', 'transformers', 'scikit-learn']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

In [ ]:
# @title 📥 Download Analyzer
import urllib.request
try:
    urllib.request.urlretrieve('https://raw.githubusercontent.com/hincaltopcuoglu/Npath-text-mining/master/distilbert_pattern_analyzer.py', 'distilbert_pattern_analyzer.py')
    from distilbert_pattern_analyzer import DistilBertPatternAnalyzer
    print('✅ Ready!')
except: print('❌ Download failed')

In [ ]:
# @title 📤 Upload Dataset
from google.colab import files
print('Upload opinions.csv')
uploaded = files.upload()
if 'opinions.csv' in uploaded: print('✅ Uploaded!')
else: print('❌ Not found')

In [ ]:
# @title 🎯 Run Analysis
import pandas as pd
from distilbert_pattern_analyzer import DistilBertPatternAnalyzer

df = pd.read_csv('/content/opinions.csv', sep=',', quotechar='"', escapechar='\\', on_bad_lines='skip', engine='python')
df.columns = df.columns.str.replace(';;;;;;', '')
df = df.dropna(subset=['text', 'type'])
df = df[df['text'].str.len() > 10]
df = df[df['type'].str.len() > 0]

sample_size = 500
texts = df['text'].head(sample_size).tolist()
categories = df['type'].head(sample_size).tolist()

analyzer = DistilBertPatternAnalyzer(results_dir='/content/distilbert_results')
analyzer.extract_embeddings_and_attention(texts, batch_size=32)
analyzer.extract_semantic_themes(n_themes=5, n_sub_themes=3)
analyzer.map_texts_to_themes(texts, categories)
analyzer.create_sankey_diagram('distilbert_patterns_sankey.html')
df_patterns = analyzer.export_patterns_to_csv('distilbert_patterns.csv')
print('✅ Done!')

In [ ]:
# @title 🌊 View Sankey
from IPython.display import IFrame
display(IFrame(src='/content/distilbert_results/distilbert_patterns_sankey.html', width=1600, height=1000))

In [ ]:
# @title 📊 Analyze
for category in sorted(df_patterns['category'].unique()):
    print(f'{category}: {len(df_patterns[df_patterns["category"]==category])} texts')

In [ ]:
# @title 📥 Download
from google.colab import files
files.download('/content/distilbert_results/distilbert_patterns_sankey.html')
files.download('/content/distilbert_results/distilbert_patterns.csv')